# Análisis exploratorio: anuncios de venta de coches

En este notebook hago una exploración básica del conjunto de datos `vehicles_us.csv`, con anuncios de coches usados en Estados Unidos. El objetivo no es un análisis profundo, sino conocer los datos y decidir qué gráficos vale la pena mostrar en la aplicación web (`app.py`).

## 1. Carga de datos

In [1]:
import pandas as pd
import plotly.express as px

car_data = pd.read_csv('../vehicles_us.csv')
car_data.head()

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
0,9400,2011.0,bmw x5,good,6.0,gas,145000.0,automatic,SUV,NaN,1.0,2018-06-23,19
1,25500,NaN,ford f-150,good,6.0,gas,88705.0,automatic,pickup,white,1.0,2018-10-19,50
2,5500,2013.0,hyundai sonata,like new,4.0,gas,110000.0,automatic,sedan,red,NaN,2019-02-07,79
3,1500,2003.0,ford f-150,fair,8.0,gas,NaN,automatic,pickup,NaN,NaN,2019-03-22,9
4,14900,2017.0,chrysler 200,excellent,4.0,gas,80903.0,automatic,sedan,black,NaN,2019-04-02,28


In [2]:
car_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  str    
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   42258 non-null  str    
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 7.7 MB


## 2. Valores ausentes y duplicados

In [3]:
missing = car_data.isna().sum()
missing_pct = (missing / len(car_data) * 100).round(1)
pd.DataFrame({'ausentes': missing, '% del total': missing_pct})[missing > 0]

,ausentes,% del total
model_year,3619,7.0
cylinders,5260,10.2
odometer,7892,15.3
paint_color,9267,18.0
is_4wd,25953,50.4


In [4]:
print('Filas duplicadas:', car_data.duplicated().sum())

Filas duplicadas: 0


Hay valores ausentes en cinco columnas: `model_year` (7.0%), `cylinders` (10.2%), `odometer` (15.3%), `paint_color` (18.0%) e `is_4wd` (50.4%). No hay filas duplicadas.

En `is_4wd` la columna solo contiene `1` o vacío, así que interpreto los vacíos como "no tiene tracción 4x4". En `paint_color` los vacíos los marco como `desconocido`. En `model_year`, `cylinders` y `odometer` no relleno nada: prefiero dejar los vacíos fuera de los gráficos que inventar valores.

## 3. Limpieza ligera

In [5]:
car_data['date_posted'] = pd.to_datetime(car_data['date_posted'])
car_data['is_4wd'] = car_data['is_4wd'].fillna(0).astype(int)
car_data['paint_color'] = car_data['paint_color'].fillna('desconocido')
car_data['manufacturer'] = car_data['model'].str.split().str[0]
car_data[['price', 'model_year', 'odometer', 'days_listed']].describe()

,price,model_year,odometer,days_listed
count,51525.000000,47906.000000,43633.000000,51525.00000
mean,12132.464920,2009.750470,115553.461738,39.55476
std,10040.803015,6.282065,65094.611341,28.20427
min,1.000000,1908.000000,0.000000,0.00000
25%,5000.000000,2006.000000,70000.000000,19.00000
50%,9000.000000,2011.000000,113000.000000,33.00000
75%,16839.000000,2014.000000,155000.000000,53.00000
max,375000.000000,2019.000000,990000.000000,271.00000


En el resumen se ven valores extremos: hay precios muy bajos (1,069 anuncios cuestan menos de \$500), precios muy altos (17 anuncios pasan de \$100,000) y odómetros enormes (46 anuncios superan las 400,000 millas). Por eso, en la app agrego filtros para acotar el rango de precio y de año.

## 4. Distribución del odómetro

In [6]:
fig = px.histogram(car_data.dropna(subset=['odometer']), x='odometer', nbins=60,
                   labels={'odometer': 'Odómetro (millas)'})
fig.update_layout(title_text='Distribución del odómetro')
fig.show()

La mayoría de los coches está entre 70,000 y 155,000 millas (la mediana es de 113,000), con una cola larga hacia la derecha por unos pocos coches con kilometraje muy alto.

## 5. Distribución del precio por condición

In [7]:
fig = px.histogram(car_data, x='price', color='condition', nbins=60,
                   labels={'price': 'Precio (USD)', 'condition': 'Condición'})
fig.update_layout(title_text='Distribución del precio por condición')
fig.show()

In [8]:
car_data.groupby('condition')['price'].median().sort_values(ascending=False)

condition
new          21999.0
like new     13995.0
excellent    10495.0
good          7900.0
fair          2500.0
salvage       2500.0
Name: price, dtype: float64

El precio sube con la condición: la mediana de los coches `new` es de \$21,999 y la de `like new` de \$13,995, mientras que `fair` y `salvage` quedan en \$2,500.

## 6. Precio vs. odómetro

In [9]:
sample = car_data.dropna(subset=['odometer']).sample(5000, random_state=42)
fig = px.scatter(sample, x='odometer', y='price', color='condition', opacity=0.6,
                 labels={'odometer': 'Odómetro (millas)', 'price': 'Precio (USD)', 'condition': 'Condición'})
fig.update_layout(title_text='Precio vs. odómetro (muestra de 5,000 anuncios)')
fig.show()

In [10]:
car_data[['price', 'odometer', 'model_year', 'cylinders', 'days_listed']].corr()['price'].round(2)

price          1.00
odometer      -0.42
model_year     0.43
cylinders      0.30
days_listed   -0.00
Name: price, dtype: float64

A más millas, el precio tiende a bajar, aunque la relación es moderada (correlación de -0.42). El año del modelo pesa casi igual pero en sentido contrario (0.43), y el número de cilindros tiene una relación más débil (0.30). Los días que un anuncio estuvo publicado no muestran relación con el precio.

## 7. Precio por tipo de vehículo

In [11]:
fig = px.box(car_data, x='type', y='price',
             labels={'type': 'Tipo de vehículo', 'price': 'Precio (USD)'})
fig.update_layout(title_text='Precio por tipo de vehículo')
fig.show()

In [12]:
car_data.groupby('type')['price'].median().sort_values(ascending=False)

type
truck          14995.0
pickup         14200.0
coupe          12950.0
convertible    12250.0
offroad        11650.0
bus            10500.0
other           8995.0
SUV             8900.0
van             7990.0
wagon           7900.0
mini-van        6495.0
sedan           5995.0
hatchback       5980.0
Name: price, dtype: float64

Los tipos más caros según la mediana son `truck` (\$14,995) y `pickup` (\$14,200); los más baratos, `sedan` (\$5,995) y `hatchback` (\$5,980).

## 8. Conclusiones

- El conjunto tiene 51,525 anuncios y 13 columnas. Los datos ausentes se concentran en `is_4wd`, `paint_color`, `odometer`, `cylinders` y `model_year`.
- El precio depende sobre todo de la condición, el año del modelo y el kilometraje: coches más nuevos y con menos millas cuestan más.
- Los camiones y pickups son los más caros; los sedanes y hatchbacks, los más baratos.
- Hay valores extremos en precio y odómetro, por lo que la aplicación incluye filtros para que el usuario pueda acotarlos.

Con esto decidí que la app incluya: un histograma del odómetro, un histograma del precio, un gráfico de dispersión de precio contra odómetro y un diagrama de cajas por tipo de vehículo.